In [ ]:
import argparse
import torch
import os
from model_202512 import GR2ST
from dataset import SKIN, HERDataset, TenxDataset, SliceBatchSampler
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import AvgMeter, get_lr

def generate_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--batch_size', type=int, default=256)
    parser.add_argument('--max_epochs', type=int, default=201)
    parser.add_argument('--temperature', type=float, default=1.0)
    parser.add_argument('--fold', type=int, default=0)
    parser.add_argument('--dim', type=int, default=50)
    parser.add_argument('--image_embedding_dim', type=int, default=1024)
    parser.add_argument('--projection_dim', type=int, default=256)
    parser.add_argument('--heads_num', type=int, default=8)
    parser.add_argument('--heads_dim', type=int, default=64)
    parser.add_argument('--heads_layers', type=int, default=2)
    parser.add_argument('--dropout', type=float, default=0.1)
    parser.add_argument('--dataset', type=str, default='10x')
    parser.add_argument('--encoder_name', type=str, default='densenet121')
    parser.add_argument('--alpha_mse', type=float, default=50.0)
    parser.add_argument('--alpha_gate', type=float, default=1.0)
    parser.add_argument('--alpha_entropy', type=float, default=0.01)
    parser.add_argument('--spatial_radius', type=float, default=3.0)
    parser.add_argument('--conf_threshold', type=float, default=0.6)
    return parser.parse_args()

def train(model, train_dataLoader, optimizer, epoch):
    loss_meter = AvgMeter()
    tqdm_train = tqdm(train_dataLoader, total=len(train_dataLoader))
    for batch in tqdm_train:
        batch = {
            k: v.cuda() for k, v in batch.items()
            if k in ["image_features", "expression", "position", "cell_type"]
        }

        loss = model(batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        count = batch["image_features"].size(0)
        loss_meter.update(loss.item(), count)
        tqdm_train.set_postfix(train_loss=loss_meter.avg, lr=get_lr(optimizer), epoch=epoch)

def load_data(args):
    if args.dataset == 'her2st':
        print(f'load dataset: {args.dataset}')
        train_dataset = HERDataset(train=True, fold=args.fold)
        batch_sampler = SliceBatchSampler(train_dataset, args.batch_size)
        train_dataLoader = DataLoader(train_dataset, batch_sampler=batch_sampler, num_workers=0)
        test_dataset = HERDataset(train=False, fold=args.fold)
        return train_dataLoader, test_dataset, None

    elif args.dataset == 'cscc':
        print(f'load dataset: {args.dataset}')
        train_dataset = SKIN(train=True, fold=args.fold)
        train_dataLoader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=0)
        test_dataset = SKIN(train=False, fold=args.fold)
        return train_dataLoader, test_dataset, None

    elif args.dataset == '10x':
        print(f'load dataset: {args.dataset}')
        examples = ["1142243F", "CID4290", "CID4465", "CID44971", "CID4535", "1160920F"]

        datasets = [
            TenxDataset(
                image_path=f"/d/zhoujl/my_model/dataset/Alex_NatGen/{example}/{example}.tif",
                spatial_pos_path=f"/d/zhoujl/my_model/dataset/Alex_NatGen/{example}/spatial/tissue_positions_list.csv",
                reduced_mtx_path=f"/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_expression_matrices/Alex/{example}/preprocessed_matrix.npy",
                barcode_path=f"/d/zhoujl/my_model/dataset/Alex_NatGen/{example}/filtered_count_matrix/barcodes.tsv.gz",
                precomputed_features_dir=f"/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features/{example}",
                cell_type_path=f"/d/zhoujl/my_model/dataset/Alex_NatGen/spots_type/{example}/summary_spot_types.tsv",
                train=True
            )
            for example in examples
        ]

        datasets.pop(args.fold)
        print("Test name:", examples[args.fold], "Test fold:", args.fold)

        train_dataset = torch.utils.data.ConcatDataset(datasets)
        train_dataLoader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=0)
        return train_dataLoader, None, examples

    return None, None, None

def save_model(args, model, test_dataset=None, examples=None, run_idx=None):
    if args.dataset != '10x':
        save_dir = f"./model_result_202512/{args.dataset}/{test_dataset.id2name[0]}"
        os.makedirs(save_dir, exist_ok=True)
        if run_idx is None:
            save_name = f"best_{args.fold}.pt"
        else:
            save_name = f"fold_{args.fold}_run_{run_idx}.pt"
        torch.save(model.state_dict(), os.path.join(save_dir, save_name))
    else:
        save_dir = f"./model_result_202512/{args.dataset}/{examples[args.fold]}"
        os.makedirs(save_dir, exist_ok=True)
        if run_idx is None:
            save_name = f"best_{args.fold}.pt"
        else:
            save_name = f"fold_{args.fold}_run_{run_idx}.pt"
        torch.save(model.state_dict(), os.path.join(save_dir, save_name))

def main():
    args = generate_args()

    if torch.cuda.is_available():
        torch.cuda.set_device(5)

    if args.dataset == '10x':
        fold_num = 6
    elif args.dataset == 'her2st':
        fold_num = 32
    elif args.dataset == 'cscc':
        fold_num = 12
    else:
        fold_num = 1

    for i in range(fold_num):
        args.fold = i
        print(f"=== Start Fold {args.fold} ===")

        train_dataLoader, test_dataset, examples = load_data(args)
        if train_dataLoader is None:
            continue

        device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")

        model = GR2ST(
            temperature=args.temperature,
            image_dim=args.image_embedding_dim,
            spot_dim=args.dim,
            projection_dim=args.projection_dim,
            heads_num=args.heads_num,
            dropout=args.dropout,
            fusion_type='sum',
            alpha_mse=args.alpha_mse,
            alpha_gate=args.alpha_gate,
            alpha_entropy=args.alpha_entropy,
            spatial_radius=args.spatial_radius,
            conf_threshold=args.conf_threshold
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)

        for epoch in range(args.max_epochs):
            model.train()
            train(model, train_dataLoader, optimizer, epoch)

        save_model(args, model, test_dataset=test_dataset, examples=examples)
        print(f"Fold {args.fold} Finished")

if __name__ == '__main__':
    main()